In [ ]:
import os
from typing import Any, Dict, List, Tuple

import hydra
import torch
from lightning import LightningDataModule, LightningModule, Trainer
from lightning.pytorch.loggers import Logger

if os.environ.get("TOKENIZERS_PARALLELISM") is None:
    os.environ["TOKENIZERS_PARALLELISM"] = "false"

from src.utils import (
    instantiate_loggers,
)

In [ ]:
os.chdir("..")
os.getcwd()

In [ ]:
from hydra import compose, initialize
from hydra.core.hydra_config import HydraConfig

with initialize(version_base="1.3", config_path="../configs"):
    cfg = compose(
        config_name="eval.yaml",
        return_hydra_config=True,
        overrides=["experiment=alignment_v1"],  # 👈 loads configs/experiment/my_exp.yaml
    )

HydraConfig.instance().set_config(cfg)

In [ ]:
assert cfg.ckpt_path

In [ ]:
datamodule: LightningDataModule = hydra.utils.instantiate(cfg.data)

model: LightningModule = hydra.utils.instantiate(cfg.model)

In [ ]:
model

In [ ]:
logger: List[Logger] = instantiate_loggers(cfg.get("logger"))

cfg["trainer"]["max_epochs"] = 0
cfg["trainer"]["accelerator"] = "mps"
trainer: Trainer = hydra.utils.instantiate(cfg.trainer, logger=logger)

object_dict = {
    "cfg": cfg,
    "datamodule": datamodule,
    "model": model,
    "logger": logger,
    "trainer": trainer,
}

In [ ]:
trainer.strategy.connect(model)
trainer._data_connector.attach_data(model, datamodule=datamodule)

model.setup(stage="fit")

In [ ]:
path = "logs/train/runs/2026-03-15_13-15-08/checkpoints/epoch_098.ckpt"
ckpt = torch.load(path, map_location="cpu", weights_only=False)
model.load_state_dict(ckpt["state_dict"])

In [ ]:
data = datamodule.test_dataloader().dataset[2]
data

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def show_rgb(tensor, stretch=True):
    """
    tensor: numpy array or torch tensor of shape (C, H, W)
    bands: tuple of 3 indices (R, G, B)
    stretch: whether to normalize values for display
    """
    # Convert to numpy if it's a torch tensor
    if "torch" in str(type(tensor)):
        tensor = tensor.detach().cpu().numpy()

    # Select bands
    rgb = tensor[:3, :, :]

    # Move to (H, W, 3)
    rgb = np.transpose(rgb, (1, 2, 0))

    # Normalize for visualization
    if stretch:
        rgb = rgb.astype(np.float32)
        for i in range(3):
            band = rgb[:, :, i]
            band_min, band_max = band.min(), band.max()
            if band_max > band_min:
                rgb[:, :, i] = (band - band_min) / (band_max - band_min)

    plt.imshow(rgb)
    plt.axis("off")
    plt.show()


show_rgb(data["eo"]["aef"])

In [ ]:
batch = data
batch["eo"]["aef"] = batch["eo"]["aef"].unsqueeze(0)
batch["eo"]["aef"].shape

In [ ]:
geo_feats = model.geo_encoder(data).squeeze(1)

In [ ]:
concepts = [
    "Densely populated area with many houses",
    "Very sparsely populated area with few houses",
    "Arable land with crops for agriculture",
    "Pasture fields with grass for grazing animals",
    "Forested area with many trees",
    "Water bodies such as lakes, rivers and sea",
]

In [ ]:
text_tokens = model.text_encoder.processor(
    text=concepts,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=77,
)

device = model.text_encoder.device
text_tokens = {k: v.to(device) for k, v in text_tokens.items()}

text_embeds = model.text_encoder.model.get_text_features(**text_tokens)

# Project
if model.text_encoder.projector is not None:
    text_embeds = model.text_encoder.projector(text_embeds)

if model.text_encoder.extra_projector is not None:
    text_embeds = model.text_encoder.extra_projector(text_embeds)

In [ ]:
text_embeds.shape

In [ ]:
import torch
import torch.nn.functional as F


def cosine_scores(x, Y):
    """
    x: tensor of shape (1, 512)
    Y: tensor of shape (6, 512)
    returns: tensor of shape (6,)
    """
    # Normalize
    x = F.normalize(x, dim=1)  # (1, 512)
    Y = F.normalize(Y, dim=1)  # (6, 512)

    # Compute cosine similarity
    scores = torch.matmul(Y, x.T)  # (6, 1)

    return scores.squeeze(1)


cosine_scores(geo_feats, text_embeds.cpu())

In [ ]:
scores = {
    k: round(v.detach().item(), 2)
    for k, v in zip(concepts, cosine_scores(geo_feats, text_embeds.cpu()))
}
scores